In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model

# Загрузка модели
model_path = "/Users/dmitrii/Downloads/model.h5"
model = load_model(model_path)

# Подготовка данных для прогноза
def create_x_input(df_train, n_steps):
    df_input = df_train.iloc[len(df_train) - n_steps:]
    x_input = df_input.values
    return x_input

def make_predictions(x_input, x_future, points_per_call):
    predict_values = []
    x_future_len = len(x_future)
    remaining_horizon = x_future_len

    print("x_input shape before reshaping: {x_input.shape}")  # Debugging line
    
    # Ensure the input has 3 dimensions for LSTM (samples, time_steps, features)
    if len(x_input.shape) == 2:  # If it's a 2D array, reshape it
        x_input = x_input.reshape((1, x_input.shape[0], x_input.shape[1]))
    
    print("x_input shape after reshaping: {x_input.shape}")  # Debugging line

    while remaining_horizon > 0:
        current_points_to_predict = min(remaining_horizon, points_per_call)
        x_input_tensor = tf.convert_to_tensor(x_input, dtype=tf.float32)
        y_predict = model.predict(x_input_tensor, verbose=0)

        if len(y_predict.shape) == 2 and y_predict.shape[0] == 1:
            y_predict = y_predict[0]

        y_predict = y_predict[:current_points_to_predict]
        predict_values.extend(y_predict)

        for i in range(current_points_to_predict):
            cur_val = y_predict[i]
            x_input = np.delete(x_input, (0), axis=1)
            future_lag = x_future[0]
            x_future = np.delete(x_future, 0, axis=0)
            future_lag[0] = cur_val
            x_input = np.append(x_input, future_lag.reshape(1, 1, -1), axis=1)

        remaining_horizon -= current_points_to_predict

    return predict_values


# Пример использования
x_input = create_x_input(df_train, lag)
x_future = df_test_no_lcr.values
predict_values = make_predictions(x_input, x_future, points_per_call)

# Обработка и сохранение результатов
df_forecast['P_l'] = predict_values
df_forecast = replace_zeros_with_average(df_forecast, 'P_l')

if len(diff_cols) > 0:
    for col in diff_cols:
        df_forecast[col] = df_true_all_col[col]

df_forecast[col] = df_true_all_col[col]
df_comparative = tn.df_denormalize_with_meta(df_forecast, min_val, max_val)

df_predict = df_comparative.copy()
df_predict = df_predict[["P_l", "time"]]
path = f"{experiment_dir}/predict.xlsx"
df_predict.to_excel(path, index=False)

# Отрисовка и метрики
fig_p_l = make_subplots(rows=1, cols=1, subplot_titles=['P_l_real vs P_l_predict'])

fig_p_l.add_trace(
    go.Scatter(x=df_true['time'], y=df_true['P_l'], mode='lines', name='P_l_real', line=dict(color='blue')), row=1,
    col=1)
fig_p_l.add_trace(go.Scatter(x=df_comparative['time'], y=df_comparative['P_l'], mode='lines', name='P_l_predict',
                             line=dict(color='orange')), row=1, col=1)
template = "presentation"

fig_p_l.update_layout(template="presentation")

output_path = "/Users/dmitrii/Downloads/real_vs_predict.html"
fig_p_l.write_html(output_path)

y_true = df_true['P_l']
y_pred = df_comparative['P_l']

rmse, r2, mae, mape, wmape = calculate_metrics(y_true=y_true, y_pred=y_pred)

print(f'MAPE = {mape}')

metrix_dict = {
    "RMSE": rmse,
    "R-squared": r2,
    "MAE": mae,
    "MAPE": mape,
    "WMAPE": wmape
}

res_dict[experiment_dir] = mape

df_metrics = pd.DataFrame(list(metrix_dict.items()), columns=['Metric', 'Value'])

output_path = f"/Users/dmitrii/Downloads/metrics.xlsx"
df_metrics.to_excel(output_path, index=False)